# RiskLens: CECL production workflow

This notebook is the **traditional CECL run**: the process a bank's allowance team runs every month-end, before any AI is involved.

| Step | What happens | Where |
|---|---|---|
| 0 | Mount Drive, get the RiskLens code, point it at the Drive folder | Colab |
| 1 | Load the source extracts: loan and borrower snapshots, macro scenarios, scenario weights, overlays, model registry | `Risk Lens/data/` |
| 2 | Pre-run controls: data quality, model approval status, weights, macro coverage | |
| 3 | Official run: PD / LGD / EAD by assigned model version, lifetime ECL per scenario, probability weighting, overlays. Booked with a run log | `Risk Lens/data/` |
| 4 | Allowance movement analysis: waterfall, segments, macro variables, top borrowers, reconciliation | `Risk Lens/outputs/` |
| 5 | Hand-off: the same tools the RiskLens agents call | |

The **RiskLens agents never run steps 1–3**. They investigate the booked results through deterministic tools (step 5), so every number they quote traces back to this run.

**Drive layout**
```
Risk Lens/
  config/      methodology.yaml (ECL maths), models.yaml (PD/LGD scorecards), settings, overlays/story, reconciliation thresholds
  data/        source extracts + official runs (Parquet/CSV the engine reads: ecl_results.parquet, allowance_summary.csv, run_log.csv)
  databank/    readable copy for people: portfolio, borrowers and ECL results as CSV per date, reference tables, data dictionary
  outputs/     analytics cache and reports
  evaluation/  ground-truth labels for scoring the agents (never read by tools or agents)
```

**One-time setup:** the code repo is private, so Colab needs a GitHub token.
1. On GitHub: **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**. Pick *Only select repositories* → `risklens-cecl`, and under *Repository permissions* set **Contents: Read-only**.
2. In Colab: **🔑 Secrets** (left sidebar) → **Add new secret** → name `GITHUB_TOKEN`, paste the token as the value, and switch on **Notebook access**.

## 0 · Setup

In [ ]:
import os, sys, subprocess, shutil, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = Path(os.environ.get("RISKLENS_DRIVE_ROOT", "/content/drive/MyDrive/Risk Lens"))
REPO_DIR = Path(os.environ.get("RISKLENS_REPO", "/content/risklens-cecl"))
REPO_URL = "github.com/Luizkauffmann/risklens-cecl.git"
BRANCH = "main"
SYNC_CONFIG_FROM_REPO = False   # True = overwrite Drive config/ with the repo's (discards edits made in Drive)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def sh(*cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr[-2000:])
    return r.stdout

def github_token():
    # Read GITHUB_TOKEN from Colab Secrets, with a clear message if it is missing or blocked
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except userdata.SecretNotFoundError:
        raise SystemExit("Add a Colab secret named GITHUB_TOKEN (key icon in the left sidebar), then re-run this cell.")
    except userdata.NotebookAccessError:
        raise SystemExit("GITHUB_TOKEN exists but this notebook can't read it: open the key icon and switch on 'Notebook access'.")
    if not token or not token.strip():
        raise SystemExit("GITHUB_TOKEN is empty: paste the token value into the secret.")
    return token.strip()

if IN_COLAB:
    url = f"https://{github_token()}@{REPO_URL}"
    try:
        sh("git", "ls-remote", "-q", url, "HEAD")
    except RuntimeError:
        raise SystemExit("GitHub refused the token. Check it hasn't expired and that it has 'Contents: Read-only' "
                         "access to Luizkauffmann/risklens-cecl.")
    if not (REPO_DIR / ".git").exists():
        sh("git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, str(REPO_DIR))
        sh("git", "-C", str(REPO_DIR), "remote", "set-url", "origin", f"https://{REPO_URL}")  # don't keep the token on disk
    else:
        sh("git", "-C", str(REPO_DIR), "pull", "-q", url, BRANCH)
if IN_COLAB:
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "plotly")
print("code:", REPO_DIR, sh("git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s").strip())

In [ ]:
# Point RiskLens at the Drive folder (config, data, outputs, evaluation)
for sub in ("config", "data", "outputs", "evaluation"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
for f in (REPO_DIR / "config").glob("*.yaml"):
    target = DRIVE_ROOT / "config" / f.name
    if SYNC_CONFIG_FROM_REPO or not target.exists():
        shutil.copy(f, target)

os.environ.update({
    "RISKLENS_CONFIG_DIR": str(DRIVE_ROOT / "config"),
    "RISKLENS_DATA_DIR": str(DRIVE_ROOT / "data"),
    "RISKLENS_OUTPUT_DIR": str(DRIVE_ROOT / "outputs"),
    "RISKLENS_GROUND_TRUTH_DIR": str(DRIVE_ROOT / "evaluation"),
})
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.utils import config
config.reload()
from src.data.loader import DataStore
from src.pipeline import cecl_run

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_colwidth", 80)
M = lambda x: f"${x / 1e6:,.2f}M"
DATES = config.settings()["reporting_dates"]
PREV, CURR = DATES[-2], DATES[-1]
print("config  :", config.config_dir())
print("data    :", config.path("generated"))
print("periods :", DATES)

## 1 · Source data
In a bank these extracts come from the loan servicing, credit and finance systems. Here a seeded generator stands in for them. It runs only if `data/` has no loan snapshot yet, and re-running it produces identical files.

In [ ]:
seed = cecl_run.seed_source_data()
print(seed["status"], "-", seed.get("reason", seed["data_dir"]))
store = DataStore()
pd.DataFrame(cecl_run.inventory(store))

In [ ]:
# The book on each date
loans = store.loans.assign(date=store.loans.as_of_date.dt.strftime("%Y-%m-%d"))
book = loans.groupby(["portfolio", "date"]).agg(loans=("loan_id", "size"), balance=("current_balance", "sum")).unstack("date")
book.style.format({c: M for c in book.columns if c[0] == "balance"} | {c: "{:,}" for c in book.columns if c[0] == "loans"})

### Risk model registry
Each product is scored by a **PD**, an **LGD** and an **EAD** model. The registry holds versions and approval status; `models.yaml` (in `config/`) holds the scorecard parameters the engine actually uses.

In [ ]:
display(store.model_metadata)
display(store.model_assignments_at(CURR).drop(columns=["as_of_date"]))

In [ ]:
# Scorecard parameters, e.g. the CRE PD model
mp = store.model_parameters
mp[mp.model_id == "CRE_PD_v4"].set_index("parameter")[["value"]].T

### Macroeconomic scenarios (reasonable & supportable forecast, then reversion)

In [ ]:
macro = store.macro.assign(vintage=store.macro.as_of_date.dt.strftime("%Y-%m-%d"))
var = "unemployment_rate"
fig = px.line(macro, x="forecast_quarter", y=var, color="scenario", line_dash="vintage", markers=True,
              title=f"{var}: July vs August forecast vintage", height=380)
fig.show()
display(store.weights.assign(as_of_date=store.weights.as_of_date.dt.strftime("%Y-%m-%d")))
display(store.overlays.assign(as_of_date=store.overlays.as_of_date.dt.strftime("%Y-%m-%d")))

## 2 · Pre-run controls
A FAIL blocks the official run, e.g. a loan scored with `CRE_PD_v5`, which is still in validation.

In [ ]:
val = {d: cecl_run.validate_inputs(d, store) for d in DATES}
checks = pd.DataFrame({d: {c["check"]: c["status"] for c in v["checks"]} for d, v in val.items()})
print({d: v["status"] for d, v in val.items()})
checks

## 3 · Official run
Per loan: lifetime ECL = Σ marginal PD × LGD × EAD over the remaining life, for each scenario path. Then probability-weight the scenarios and add the management overlays. Both CECL (lifetime) and IFRS 9 (12-month / lifetime by stage) are booked.

In [ ]:
import time
for d in DATES:
    t = time.time()
    out = cecl_run.run_official(d)
    for r in out["runs"]:
        print(f"{r['run_id']:26s} allowance {M(r['allowance']):>10}  ({r['loan_count']:,} loans, controls {r['validation_status']})  {time.time() - t:4.1f}s")
store = DataStore()            # re-read the booked results
store.run_log[["run_id", "executed_at_utc", "allowance", "scenario_set_id", "input_hash_loans", "validation_status"]].tail(8)

In [ ]:
s = store.summary.copy()
s["as_of_date"] = s.as_of_date.dt.strftime("%Y-%m-%d")
cols = ["run_id", "loan_count", "total_balance", "modeled_ecl", "overlay_total", "allowance", "allowance_rate_pct",
        "ead_weighted_pd_12m", "ead_weighted_lgd", "scenario_ecl_baseline", "scenario_ecl_adverse", "scenario_ecl_severe_adverse"]
view = s[cols].set_index("run_id").T
view

In [ ]:
r = store.results
r = r[r.framework == "CECL"].assign(date=lambda x: x.as_of_date.dt.strftime("%Y-%m-%d"))
by_port = r.groupby(["portfolio", "date"]).ecl.sum().unstack()
by_port["change"] = by_port[CURR] - by_port[PREV]
fig = px.bar(by_port[[PREV, CURR]].reset_index().melt(id_vars="portfolio", var_name="date", value_name="ecl"),
             x="portfolio", y="ecl", color="date", barmode="group", title="Modeled CECL ECL by portfolio", height=380)
fig.show()
by_port.apply(lambda c: c.map(M))

### Databank export
A readable copy of everything in `Risk Lens/databank/`: one folder per reporting date with the loan portfolio, the borrowers and the official CECL results as CSV (they open in Google Sheets), plus the model registry, macro scenarios, weights, overlays and a data dictionary. The engine keeps reading the Parquet files in `data/`.

In [ ]:
written = cecl_run.export_databank(DRIVE_ROOT / "databank", store, frameworks=["CECL", "IFRS9"])
pd.DataFrame(written)

## 4 · Allowance movement analysis
This uses the **same deterministic tools the agents call** (`src/tools`), so the notebook, the web app and the agents all show the same numbers.

The first run computes the attribution walk and the Shapley robustness checks (about 1–2 minutes) and caches them in `outputs/cache`. Later runs load from the cache.

In [ ]:
from src.tools import call_tool, tool_schemas, list_tools, ToolCallLog
LOG = ToolCallLog(Path(config.path("outputs")) / "notebook_tool_calls.jsonl")
def tool(name, **args):
    rec = call_tool(name, args, caller="colab_workflow", log=LOG)
    if rec.status != "ok":
        raise RuntimeError(rec.error)
    return rec.result

cmp_ = tool("compare_allowance", previous_date=PREV, current_date=CURR)
print(f"Allowance {M(cmp_['previous_allowance'])} -> {M(cmp_['current_allowance'])}  "
      f"change {M(cmp_['change'])} ({cmp_['change_pct']:.1f}%)")

In [ ]:
wf = tool("calculate_portfolio_attribution", previous_date=PREV, current_date=CURR)
order = list(dict.fromkeys(s["driver"] for s in wf["steps"]))
imp = {d["driver"]: d for d in wf["drivers"]}
x = [f"{PREV}"] + [imp[d]["label"] for d in order] + [f"{CURR}"]
y = [wf["previous_allowance"]] + [imp[d]["impact"] for d in order] + [wf["current_allowance"]]
fig = go.Figure(go.Waterfall(x=x, y=y, measure=["absolute"] + ["relative"] * len(order) + ["total"],
                             text=[M(v) for v in y], textposition="outside",
                             connector={"line": {"color": "#999"}}))
fig.update_layout(title=f"CECL allowance waterfall (residual {M(wf['residual'])})", height=460, showlegend=False)
fig.show()
pd.DataFrame(wf["steps"])[["label", "driver", "impact", "allowance_after"]]

In [ ]:
seg = tool("calculate_segment_attribution", previous_date=PREV, current_date=CURR)
mat = pd.DataFrame([{"portfolio": s["segment"], **s["drivers"], "total": s["total_change"]} for s in seg["segments"]]).set_index("portfolio")
fig = px.imshow(mat.drop(columns="total") / 1e6, text_auto=".2f", color_continuous_scale="RdBu_r",
                color_continuous_midpoint=0, aspect="auto", title="Allowance change by portfolio x driver ($M)", height=380)
fig.show()
mat.apply(lambda c: c.map(M))

In [ ]:
mv = tool("calculate_variable_attribution", previous_date=PREV, current_date=CURR)   # exact Shapley, 64 projections
v = pd.DataFrame(mv["variables"])
fig = px.bar(v, x="impact", y="variable", orientation="h", title=f"Macro impact by variable (Shapley), total {M(mv['macro_total'])}", height=330)
fig.update_yaxes(categoryorder="total ascending")
fig.show()
v[["variable", "impact", "sequential_impact", "share_of_macro_pct"]]

In [ ]:
top = tool("identify_top_contributors", previous_date=PREV, current_date=CURR, n=10)
pd.DataFrame(top["contributors"])[["rank", "counterparty_id", "customer_name", "portfolio", "balance_current",
                                    "ecl_previous", "ecl_current", "ecl_change", "pd_12m_previous", "pd_12m_current",
                                    "main_driver", "reason_codes"]]

In [ ]:
mod = tool("compare_model_versions", previous_date=PREV, current_date=CURR)
w = tool("compare_scenario_weights", previous_date=PREV, current_date=CURR)
print("model versions changed:", mod["any_model_version_changed"], "| impact", M(mod["allowance_impact_of_model_change"]))
print("candidates not in production:", [m["model_id"] for m in mod["candidate_models_not_in_production"]])
print("scenario weights changed:", w["weights_changed"], "-> toward", w["weight_shifted_toward"], "| impact", M(w["allowance_impact_of_weight_change"]))

In [ ]:
rec = tool("reconcile_allowance_change", previous_date=PREV, current_date=CURR)
print(f"Reported change {M(rec['reported_change'])} | explained {M(rec['explained_change'])} | residual {M(rec['residual'])} "
      f"| {rec['pct_explained']:.2f}% explained -> {rec['overall_status']}")
display(pd.DataFrame(rec["controls"])[["control", "status", "value"]])
pd.DataFrame(rec["order_robustness"]["drivers"])[["label", "sequential", "reverse_sequential", "shapley"]]

### What-if: scenario weights
Weighted ECL is linear in the weights, so this is an exact re-weighting of the booked scenario results. It's a what-if, not a booked run.

In [ ]:
wi = tool("run_scenario_analysis", as_of_date=CURR, weights={"baseline": 0.50, "adverse": 0.30, "severe_adverse": 0.20})
print("official:", M(wi["official_allowance"]), "| 50/30/20:", M(wi["custom"]["allowance"]),
      f"({wi['custom']['difference_pct']:+.1f}%)")
{k: M(v) for k, v in wi["single_scenario_allowance"].items()}

## 5 · Hand-off to the RiskLens agents
The agents get **only these tool schemas**. Each specialist sees a focused subset (`config/agents.yaml`). Every call is validated, repaired if needed (e.g. `"Aug 2026"` → `2026-08-31`), and logged with provenance.

In [ ]:
print({a: len(list_tools(a)) for a in ["orchestrator", "portfolio_agent", "credit_agent", "macro_agent",
                                        "model_risk_agent", "reconciliation_agent"]})
r = call_tool("calculate_pd_migration", {"previous_date": "July 2026", "current_date": "aug", "portfolio": "commercial real estate"},
              caller="demo", log=LOG)
print("normalised parameters:", r.parameters)
print("provenance:", r.provenance())
bad = call_tool("calculate_pd_migration", {"previous_date": "2025-01-31", "current_date": CURR}, log=LOG)
print("structured error:", bad.error)
LOG.stats()

**Next:** the agents run on NVIDIA Brev (Nemotron 3.5 Lightning served by vLLM), pointed at a copy of this `data/` folder (`RISKLENS_DATA_DIR`).

**Try a model change:** edit `config/models.yaml` in Drive, then re-run steps 3–4. The Model Risk tools and the reconciliation will show the difference. To restore the defaults, set `SYNC_CONFIG_FROM_REPO = True`.